In [1]:
from typing import Any
from gymnasium.spaces import Sequence, Box, Tuple, Text
from gymnasium import spaces
import gymnasium as gym
import numpy as np
from numpy.char import array as chararray
import string

In [2]:
from stable_baselines3.ppo.policies import MlpPolicy
from stable_baselines3 import PPO, A2C
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor

In [3]:
# actions: replace bitstring[i:i+k] with the replacement string 
# rewards: defined by a utility function. evaluates the bitstring, and should be a single scalar value denoting reward? 
#   ^ can also have side effects, stored within the env
# observation: 
#   current bytestring
#   ^ e.g. coverage of lines within the program. ObsType, usually numpy array. 
# reward:
#   user defined, the overall rewawrd for the action as a single number
# reset: hidden state gets reset, pure otherwise. this matters for e.g. coverage guided fuzzing tasks

def bytestrings() -> gym.Space:
    return Sequence(Box(low=0, high=255, dtype=int), stack=True)

def transpose_1d_arr(arr_1d: np.array):
    return arr_1d[np.newaxis].T

class BitstringEnv(gym.Env):
    def __init__(
            self, 
            init_bytestring: np.array, 
            max_replacement_length: int = 20, 
            step_cost: float = 0.01):
        super(BitstringEnv, self).__init__()
        # dynamic part of the state
        self.bytestring = init_bytestring
        self.prev_utility = 0.0
         
        # constant part of the state
        self._init_bytestring = init_bytestring
        self.max_replacement_length = max_replacement_length
        self.step_cost = step_cost

    @property
    def observation_space(self):
        return bytestrings()

    @property 
    def action_space(self):
        index_space = Box(0, self.bytestring.size, dtype=int)
        replacement_len_space = Box(0, self.max_replacement_length, dtype=int)
        return Tuple(spaces=[index_space, replacement_len_space, bytestrings()])

    def reset(self, seed=None, options=None) -> tuple[np.array, dict[str, Any]]:
        super().reset(seed=seed, options=options)
        self.bytestring = self._init_bytestring
        self.prev_utility = 0.0
        return transpose_1d_arr(self.bytestring), {}  # empty info dict

    def utility(self) -> float:
        '''Evaluates the current state.
        Abstract method. 
        '''
        return 0.0

    def step(self, action) -> tuple[np.array, float, bool, bool, dict[str, Any]]:
        '''Performs the action on the state: a string replacement at the chosen index.
        '''
        # applying action on the state
        repl_start = action[0][0]
        repl_end = repl_start + action[1][0]
        repl_str = action[2].T[0].astype(int)
        self.bytestring = np.concatenate((self.bytestring[:repl_start], repl_str, self.bytestring[repl_end:]))

        # Never terminate or limit the number of steps.
        terminated = False
        truncated = False

        # Reward based on the utility. Each step has a default negative punishment.
        current_utility = self.utility()
        reward = current_utility - self.prev_utility - self.step_cost
        self.prev_utility = current_utility

        return (
            transpose_1d_arr(self.bytestring),
            reward, # 1 or 0
            terminated, # bool
            truncated, # bool
            {}, # extra info, dict
        )

    def render(self) -> None:
        # print string representing the environment
        print(self.bytestring)

    def close(self):
        pass

In [32]:
def bytestrings_fixed(l: int) -> gym.Space:
    # Sequence(Box(low=0, high=255, dtype=int), stack=True)
    # return Text(max_length=l, charset=string.hexdigits)
    return Box(low=0, high=256, shape=(l,), dtype=int)
    
def pad_to(a: np.array, l : int) -> np.array:
    '''Returns an array of length `l`, which is `a` padded to exactly
    `l` elements using zeros.
    '''
    e = np.zeros(shape=(l,), dtype=int)
    pad_len = min(e.size, a.size)
    e[:pad_len] = a[:pad_len]
    return e

def from_padded(a: np.array, pad_elem: int = 0) -> np.array:
    zero_idxs = np.transpose(np.nonzero(a == pad_elem))
    if zero_idxs.size == 0:
        # all doesn't equal `pad_elem` => there are no padding
        return a
    # slice the array until the first zero
    return a[:zero_idxs[0][0]]

def size_of_padded(a: np.array, pad_elem: int = 0) -> int:
    return from_padded(a, pad_elem).size

class BitstringEnvFixed(gym.Env):
    def __init__(
            self, 
            init_bytestring: np.array, 
            max_bytestring_len: int = 1000,
            max_replacement_length: int = 20, 
            step_cost: float = 0.01):
        super().__init__()
         
        # constant part of the state
        self._init_bytestring = pad_to(init_bytestring, max_bytestring_len)
        self.max_bytestring_len = max_bytestring_len
        self.max_replacement_length = max_replacement_length
        self.step_cost = step_cost

        # dynamic part of the state
        self.bytestring = self._init_bytestring
        self.prev_utility = 0.0

    @property
    def observation_space(self):
        return bytestrings_fixed(self.max_bytestring_len)

    @property
    def action_space(self):
        index_space = Box(0, size_of_padded(self.bytestring), dtype=int)
        replacement_len_space = Box(0, self.max_replacement_length, dtype=int)
        return Tuple(spaces=[
            index_space, replacement_len_space, bytestrings_fixed(self.max_replacement_length)
            ])

    def reset(self, seed=None, options=None) -> tuple[np.array, dict[str, Any]]:
        super().reset(seed=seed, options=options)
        self.bytestring = self._init_bytestring
        self.prev_utility = 0.0
        return self.bytestring, {}  # empty info dict

    def utility(self) -> float:
        '''Evaluates the current state.
        Abstract method. 
        Should be continuous in the sequence, and measure the distance of the current
        value to the "ideal" one.
        '''
        return 0.0

    def step(self, action) -> tuple[np.array, float, bool, bool, dict[str, Any]]:
        '''Performs the action on the state: a string replacement at the chosen index.
        '''
        # applying action on the state
        repl_start = action[0][0]
        repl_end = repl_start + action[1][0]
        repl_str = from_padded(action[2])
        self.bytestring = pad_to(
              np.concatenate([
                self.bytestring[:repl_start], repl_str, self.bytestring[repl_end:]])
            , self.max_bytestring_len)

        # Reward based on the utility. Each step has a default negative punishment.
        current_utility = self.utility()
        reward = 0 - current_utility - self.step_cost
        self.prev_utility = current_utility

        terminated = current_utility == 0
        truncated = False

        return (
            pad_to(self.bytestring, self.max_bytestring_len),
            reward,
            terminated, # bool
            truncated, # bool
            {}, # extra info, dict
        )

    def render(self) -> None:
        # print string representing the environment
        print(from_padded(self.bytestring))

    def close(self):
        pass

class HammingEnv(BitstringEnvFixed):
    @staticmethod
    def hamming_d(str_a: np.array, str_b: np.array) -> float:
        length_d = abs(str_a.size - str_b.size)
        compare_upto = str_a.size
        if length_d != 0:
            pass
            compare_upto = min(str_a.size, str_b.size)

        mismatches = 0
        for i in range(compare_upto):
            mismatches += 1 if str_a[i] != str_b[i] else 0

        return length_d + mismatches

    def utility(self) -> float:
        '''Distance function: dist(str_a, str_b) = | len(a) - len(b) | + | mismatches |
        '''
        target = np.array([7] * 30 + [1] * 5 + [2] * 10 + [3] * 10 + [5] * 9, dtype=int)
        source = from_padded(self.bytestring)
        return HammingEnv.hamming_d(source, target)

In [33]:
class FlattenAction(gym.ActionWrapper):
    """Action wrapper that flattens the action."""
    def __init__(self, env):
        super(FlattenAction, self).__init__(env)
        self.action_space = gym.spaces.utils.flatten_space(self.env.action_space)
        
    def action(self, action):
        return gym.spaces.utils.unflatten(self.env.action_space, action)

    def reverse_action(self, action):
        return gym.spaces.utils.flatten(self.env.action_space, action)

In [34]:
env = Monitor(
    gym.wrappers.TimeLimit(
    FlattenAction(
        HammingEnv(init_bytestring=np.array([1] * 40 + [2] * 15))),
    100
    ))
check_env(env)

/home/jacob/.local/lib/python3.10/site-packages/stable_baselines3/common/env_checker.py:462: UserWarning: We recommend you to use a symmetric and normalized Box action space (range=[-1, 1]) cf. https://stable-baselines3.readthedocs.io/en/master/guide/rl_tips.html
  warnings.warn(
/home/jacob/.local/lib/python3.10/site-packages/stable_baselines3/common/env_checker.py:473: UserWarning: Your action space has dtype int64, we recommend using np.float32 to avoid cast errors.
  warnings.warn(


In [ ]:
# TODO: make everything based off floating points, then round to naturals.
# TODO: normalise the action space 
# hand crank the model, see if the action it picks makes sense
# then update the environment, check if the environment is updating properly
# run the model for a few steps if all looks good and see if it approaches the goal.
# DONE write a loop that will run the agent on the environment, see the results rendered.
# DONE perform this on a freshly trained environment.
# to make training more effective, start with a vector of environments with a random initial vector?
# loss is increasing as training happens, learning rate too high?

In [38]:
# vec_env = make_vec_env(HammingEnv, n_envs=1, env_kwargs=dict(init_bytestring=np.array([0] * 50 + [1] * 14, dtype=int)))
# vec_env = make_vec_env(GoLeftEnv, n_envs=1, env_kwargs=dict(grid_size=10))
model = PPO(
    MlpPolicy, 
    env, 
    verbose=1, 
    learning_rate=3e-4,
    policy_kwargs=dict(net_arch=[256, 256]), 
    tensorboard_log="./logs/ppo_mlp_int_repr")
model.learn(total_timesteps=100_000, tb_log_name="ppo_arch256_rate3e-4")

Using cpu device
Wrapping the env in a DummyVecEnv.
Logging to ./logs/ppo_mlp_int_repr/ppo_arch256_rate3e-4_1


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 100      |
|    ep_rew_mean     | -5.4e+03 |
| time/              |          |
|    fps             | 1286     |
|    iterations      | 1        |
|    time_elapsed    | 1        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 100         |
|    ep_rew_mean          | -5.43e+03   |
| time/                   |             |
|    fps                  | 841         |
|    iterations           | 2           |
|    time_elapsed         | 4           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.010488932 |
|    clip_fraction        | 0.138       |
|    clip_range           | 0.2         |
|    entropy_loss         | -31.2       |
|    explained_variance   | -3.87e-05   |
|    learning_rate        | 0.

In [ ]:
ob, _ = env.reset()
print("initial:")
env.render()

for i in range(2):
    act, _ = model.predict(ob)
    ob_alt = apply_action(ob, act)
    ob, rwd, trm, tnc, _ = env.step(act)
    print(f"step {i+1}: applied {act=}, reward {rwd=}")

    all_equal = np.all(ob_alt == ob)
    print(f"{all_equal=}")
    if not all_equal:
        print(f"alt: {from_padded(ob_alt)}")
        print(f"obs: {from_padded(ob)}")

initial:
[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2]
step 1: applied act=array([0.10257327, 0.07382357, 0.        , 0.        , 1.20568359,
       0.03444609, 0.17745349, 2.67771196, 0.        , 0.43548638,
       0.07682696, 0.        , 0.        , 0.13434157, 0.80984795,
       0.        , 1.91035235, 0.        , 0.28344002, 0.        ,
       0.        , 0.11521573]), reward rwd=-54.01
all_equal=np.True_
step 2: applied act=array([0.3202765 , 0.29932821, 0.        , 0.        , 0.        ,
       0.42966378, 1.97826385, 2.18334031, 0.03569396, 0.00390673,
       0.        , 0.        , 0.8718617 , 0.        , 0.        ,
       1.94062364, 0.79490095, 0.        , 0.        , 0.2503913 ,
       0.89177591, 0.78610456]), reward rwd=-54.01
all_equal=np.True_


In [ ]:
def apply_action(obs: np.array, action: np.array) -> np.array:
    action_int = np.floor(action).astype(int) # floor is used instead of round
    mut_idx, mut_substr_length = action_int[:2]
    mut_replacement = from_padded(action_int[2:])
    mut_obs = pad_to(
          np.concatenate([obs[:mut_idx], mut_replacement, obs[mut_idx+mut_substr_length:]])
        , 1000)
    return mut_obs

In [ ]:
target_arr = pad_to(np.array([1] * 40 + [2] * 15), 1000)
obs_init = pad_to(np.array([7] * 30 + [1] * 5 + [2] * 10 + [3] * 10 + [5] * 9), 1000)
obs_2 = pad_to(np.array([3] * 40 + [2] * 15), 1000)
obs_3 = pad_to(np.array([1] * 39 + [100] + [2] * 15), 1000)

action, _ = model.predict(obs_3)
mut_idx, mut_substr_length = action[:2]
mut_addition = action[2:]
print(f"{mut_idx=} {mut_substr_length=}")
print(f"{mut_addition=}")

new_obs = apply_action(obs_3, action)
print(f"obs_3   = {from_padded(obs_3)}")
print(f"new_obs = {from_padded(new_obs)}")

new_obs_real, rwd, term, trnc, info = env.step(action)
print(f"new_obs_env = {from_padded(new_obs_real)}")


mut_idx=np.float64(0.0) mut_substr_length=np.float64(0.0)
mut_addition=array([0.        , 0.43085319, 0.        , 0.81712788, 0.        ,
       0.        , 2.40206909, 0.        , 2.51411057, 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ])
obs_3   = [  1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1
   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1
   1   1   1 100   2   2   2   2   2   2   2   2   2   2   2   2   2   2
   2]
new_obs = [  1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1
   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1
   1   1   1 100   2   2   2   2   2   2   2   2   2   2   2   2   2   2
   2]
new_obs_env = [2 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2 2 2 2 2]


In [ ]:
print(f"input obs: {from_padded(obs_3)}")
env.render()
action, _ = model.predict(obs_3)
print(f"applying: {action=}")
obs, rew, trm, tnc, _ = env.step(action)
env.render()

input obs: [  1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1
   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1   1
   1   1   1 100   2   2   2   2   2   2   2   2   2   2   2   2   2   2
   2]
[  1   1   2   1   1   1   1   1   1   1 173 173 221 248 159  54 219 242
  31 162  33   4  90  74 190 233 245  48 207 199 126  13 139 178 209 218
 169 160 147   2  71  63 214 254 193 213  86 100 225  36 222  47 149   3
 181  53 208  81 252  86 211 182 151  97 111 176 141  98 132 175  43  20
 151  36 168 218  74 239 210  74  35  97 207 256   2 107 155  29  53 240
  96 182 230 153 180  75  50  71  47 243 116  53  91 119 123 231  76   1
   1   1   1   1   1   1   1 204  23 128  22  49  78 102 199  19  55  41
  80  15 220 157  59 135 135 109 200   2   2   2   2   2   2   2   2   2
   2   2]
applying: action=array([0.3663674 , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       1.41130352, 

In [39]:
evaluate_policy(model, env)

(np.float64(-5401.0), np.float64(0.0))

In [ ]:
a1 = pad_to(np.array([7] * 30 + [1] * 5 + [2] * 10 + [3] * 10 + [5] * 9), 1000)
a2 = pad_to(np.array([1] * 40 + [2] * 15), 1000)
HammingEnv.hamming_d(a1, a2)

54